[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/01_react.ipynb)

# Part 1 — ReAct

An **agent** is a large language model running in a loop with the ability to access **tools**; ordinary Python functions it can ask you to run on its behalf.

The model never executes anything itself. Each turn it reads the conversation so far and replies with one of two things: a final answer, or a request to *call this function with these arguments*. Your code runs the function and sends the result back. The model reads that result and decides what to do next, and the process repeats until reaching an exit condition.

The `prompt`+`tool call` loop is called **ReAct**, short for Reason + Act ([Yao et al. 2022](https://arxiv.org/abs/2210.03629)). It is the simplest agent design there is, and we implement it below. By using a dictionary to map tool names to Python functions, we can directly write a `for` loop that implements the paper.

**What does this notebook do?** We implement the ReAct framework in about 30 lines of code. First we provide the syntax to specify a tool via a JSON header, then we construct the ReAct loop, and finally we make a simple calculator agent that ensures an LLM performs arithmetic through Python, rather than attempting to "memorize" the answer to arithmetic operations from its training data.

This last task is a simple (but crucial!) example of why we use agents. They allow us to offload critical computation onto trusted, conventional software.

**You are done when** the agent computes 13 × 47 + 8 by calling a calculator tool and reports the answer 619.

## 1.0 API Setup

Same setup as Part 0. If set up correctly, a print statement will tell you the model has loaded.

In [ ]:
# Setup -- same as 00_api_access.ipynb
import os, sys, time, json
if 'google.colab' in sys.modules:
    %pip install -U -q "google-genai<2.13" "google-auth==2.49.0"

from google import genai
from google.genai import types as gtypes

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.1-flash-lite"


def generate_with_retry(*, contents, config=None, max_attempts=6):
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)

print(f"Gemini client ready (model={MODEL}).")

## 1.1 A tool is a function plus a schema

A tool consists of two pieces. The first is an ordinary Python function. The second is a **JSON schema**. This contains: the function's name; a plain text description of what it does; and what arguments it accepts.

This schema represents an interface between the LLM and the tool. The LLM has no access to the source code itself and relies entirely on the context from the function name and description to infer how it is meant to interact with the tool.

Two parts of the schema do most of the work:

- `name` and `description` decide whether the model picks this tool over another. Write them for a colleague who cannot read the code.
- `enum` restricts a value to a fixed set. It steers what the model supplies, which is why the function below still checks the value before trusting it.

The code block below implements a simple calculator tool. Its `op` argument selects one of four operations: addition, subtraction, multiplication or division.

In [ ]:
def calculator(op: str, a: float, b: float) -> dict:
    """Perform one arithmetic operation."""
    ops = {"add": a + b, "sub": a - b, "mul": a * b,
           "div": a / b if b != 0 else float('nan')}
    if op not in ops:
        return {"error": f"unknown op {op!r}; valid: add, sub, mul, div"}
    return {"result": ops[op]}


CALC_PARAMS = {
    "type": "object",
    "properties": {
        "op": {"type": "string", "enum": ["add", "sub", "mul", "div"],
               "description": "operation to perform"},
        "a":  {"type": "number"},
        "b":  {"type": "number"},
    },
    "required": ["op", "a", "b"],
}

def calc_schema():
    return gtypes.FunctionDeclaration(
        name="calculator",
        description="A simple tool to evaluate one arithmetic operation using the Python standard library.",
        parameters=CALC_PARAMS)

print(json.dumps(CALC_PARAMS, indent=2))

## 1.2 The ReAct loop

We now have what we need to implement ReAct in a `for` loop. The `run_agent` function below repeats three steps:

1. Send the conversation so far, plus the tool schemas, to the model.
2. If the reply contains a `function_call`, look the name up in `tool_fns`, run that function, and append its result to the conversation as an **observation**.
3. If the reply is plain text instead, that is the final answer, and the loop stops.

`max_steps` provides a limit to the number of steps - this is important in case something goes wrong; we don't want to blow through our token budget!

Every tool call is wrapped in `try/except`. If the model sends a bad argument, the exception text goes back to it as the observation, and it usually corrects itself on the next turn — so a failed call does not stop the run. Watch for this when you run the cells below.

In [ ]:
def run_agent(system_prompt, user_prompt, tool_schemas, tool_fns,
              max_steps=12, temperature=0.2, verbose=True):
    """Minimal ReAct loop. Returns (final_text, transcript)."""
    contents = [gtypes.Content(role="user",
                               parts=[gtypes.Part.from_text(text=user_prompt)])]
    cfg = gtypes.GenerateContentConfig(
        system_instruction=system_prompt,
        tools=[gtypes.Tool(function_declarations=tool_schemas)],
        temperature=temperature,
        automatic_function_calling=gtypes.AutomaticFunctionCallingConfig(disable=True))

    transcript = []
    for step in range(max_steps):
        resp = generate_with_retry(contents=contents, config=cfg)
        parts = resp.candidates[0].content.parts or []
        contents.append(resp.candidates[0].content)

        calls = [p.function_call for p in parts if getattr(p, "function_call", None)]
        if not calls:
            text = "".join(getattr(p, "text", "") or "" for p in parts)
            transcript.append(("final", text))
            if verbose: print(f"[{step}] FINAL: {text[:160]}")
            return text, transcript

        obs = []
        for fc in calls:
            name, args = fc.name, dict(fc.args or {})
            if verbose: print(f"[{step}] CALL {name}({args})")
            try:
                result = tool_fns[name](**args)
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}   # errors are observations
            transcript.append((name, args, result))
            if verbose: print(f"[{step}]   -> {json.dumps(result)[:160]}")
            obs.append(gtypes.Part.from_function_response(name=name, response=result))
        contents.append(gtypes.Content(role="user", parts=obs))

    transcript.append(("final", "(max_steps reached)"))
    return "(max_steps reached)", transcript

## 1.3 A task to run it on

A common "gotcha" of early LLMS that wowed the social media world was to ask it the answer to `2+2` or to count the number of R's in "strawberry". To understand why LLMs fail at these tasks, recall that they are, at the core, doing pattern matching against their training dataset. If an LLM was trained on all of the internet, asking it to evaluate `2+2` is telling you whether it's memorized how often it saw `2+2=4`. By offloading to a tool, we are swapping a statistical inference task for a floating point calculation.

To highlight this, we will ask our agent to **compute 13 × 47 + 8 using the calculator tool.** The answer is, of course, 619.

*Note:* The system prompt tells the model never to do arithmetic in its head, and that is what forces the tool calls. Try deleting that instruction later and see what changes.

In [ ]:
TOY_SYSTEM = ("You are a careful arithmetic assistant. Use the calculator tool for every "
              "computation; never do arithmetic in your head.")
TOY_USER = "Compute 13 * 47 + 8. Return just the final number."

answer, transcript = run_agent(TOY_SYSTEM, TOY_USER,
                               tool_schemas=[calc_schema()],
                               tool_fns={"calculator": calculator},
                               max_steps=8)
print("\n=== Final answer:", answer)

## 1.4 When to reach for ReAct

ReAct is the right job for most tasks. In fact, it is (for the most part) what is running behind the scenes in Claude. In general, you want ReAct when:

- you cannot write the procedure down in advance, because each step depends on what the last one returned;
- a wrong step is cheap to undo;
- there are only a handful of tools to choose between.

After we give a crash course on the other frameworks available to us, we will return to ReAct as the right tool to perform an adaptive finite element analysis. In that example, the number of refinement cycles a mesh requires depends on error the agent has not measured yet, and in small dimensions it is relatively cheap to throw out a poorly generated mesh and try again.

ReAct gets harder to rely on in three situations: when some step must *always* happen, when different parts of the job need genuinely different expertise, and when a wrong action is expensive. These will motivate the other frameworks that we consider next.

## 1.5 Further reading

- **ReAct: Synergizing Reasoning and Acting in Language Models** — [Yao et al., 2022](https://arxiv.org/abs/2210.03629). The paper this loop comes from; sections 1–3 are the readable part.
- **[Gemini function calling](https://ai.google.dev/gemini-api/docs/function-calling)** — the reference for what we used here, including parallel calls and the automatic mode we switch off in `run_agent`.
- **[Understanding JSON Schema](https://json-schema.org/understanding-json-schema)** — the format tool schemas are written in. The `type` and `enum` sections are the relevant ones.
- **Tool calling in the other APIs** — [Anthropic](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview) · [OpenAI](https://developers.openai.com/api/docs). The same three ideas with different names.
- **[The ReAct pattern, illustrated](https://www.promptingguide.ai/techniques/react)** — a shorter walk-through if the paper is heavy going.